In [ ]:
#DataScienceCourse/
#├── Background/               # Cheatsheets and reference manuals
#├── docs/                     # Exported Markdown notebooks for GitHub Pages
#├── notebooks/
#│   ├── 00_ProjectTemplate.ipynb
#│   └── SX.Y_DESC.ipynb       # Sequential step notebooks
#├── persisted/                # Final serialized joblib artifacts
#│   ├── descriptive/
#│   ├── models/
#│   └── framework/
#├── src/                      # Flat Python package (auto-discovered by setuptools)
#│   ├── __init__.py
#│   ├── paths.py              # OS-agnostic relative path resolution
#│   ├── descriptive_tools.py  # Statistical profiling & ESS metadata helpers
#│   ├── s0_screening.py       # Step 0: Exploratory screening & ESS cataloging
#│   ├── s1_ingestion.py       # Step 1: APIs, scraping & SQL persistence
#│   ├── s2_cleaning.py        # Step 2: Survey sanitization, macro merge & audit flags
#│   ├── s3_eda.py             # Step 3: Exploratory stats persistence
#│   ├── s4_features.py        # Step 4: Deterministic feature engineering
#│   ├── s5_modeling.py        # Step 5: GroupKFold modeling & benchmarking
#│   ├── wrapper.py            # Step 6.1: Global orchestrator & ablation runner
#│   ├── apps_dash/            # Standalone Plotly Dash sandbox
#│   ├── apps_fapi/            # Standalone FastAPI sandbox
#│   └── apps_flask/           # Standalone Flask sandbox
#├── temp/                     # Transitional checkpoints, raw HTML/XML (gitignored)
#├── tests/                    # Pytest data-contract & pipeline tests
#├── pyproject.toml            # Single source of truth for dependencies & CLI
#├── ProjectStatement.md       # Technical specification
#└── Index_Readme.md           # Sequential workflow guide
import sys
from pathlib import Path

root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from paths import setup_workspace

PATHS = setup_workspace()

In [ ]:
# ==============================================================================
# NOTEBOOK SCOPE:
# 
# [ACTIVE] STEP 0.0 — Codebook Parsing & Metadata Configuration (L0): 
#                     Parse the ESS11 HTML/XML Codebook into valid_cols.json 
#                     and blacklist.json, 
#                     defining domain sectors, scale bounds, 
#                     non-response masks, and COALESCE pairs, 
#                     and persist metadata into DuckDB/SQLite.
# ==============================================================================

In [ ]:
# ==============================================================================
# LAYER 0: ESS DATA DICTIONARY INGESTION (build_db.py)
# ------------------------------------------------------------------------------
# HF Repo (Hugging face public 
#         repository"https://huggingface.co/datasets/JBG89Spain/ESSsurveys.sav")
#  ├── metadata.html → is downloaded (small)  (L0)
#  │         │
#  │         ▼
#  │        dicts → filter → columns
#  │
#  └── data.csv → It is NOT downloaded
#            │
#            ▼
#       DuckDB reads it by URL  (L1)
#       SELECT only filtered columns(L0 dict)
#            │
#            ▼
#       .temp/result.parquet   
# ------------------------------------------------------------------------------
#
# ==============================================================================
# ==============================================================================
# CONSTANTS (always defined before any function)
# ==============================================================================
# ==============================================================================
# L0  Step A) Download html
# ==============================================================================
import json
import re
import subprocess
import shutil
import textwrap
from collections import defaultdict
from pathlib import Path
from bs4 import BeautifulSoup
from paths import setup_workspace



PATHS = setup_workspace()
TABLES: Path = PATHS["TABLES"]

HTML_PATH = TABLES / "ESSCodebook.html"
DICT_PATH = TABLES / "valid_cols.json"
BLACKLIST_PATH = TABLES / "blacklist.json"
URL = "https://huggingface.co/datasets/JBG89Spain/ESSsurveys.sav/resolve/main/ESSCodebook.html"

COUNTRY_CODES = {
    "al", "at", "be", "bg", "ch", "cy", "cz", "de", "dk", "ee", "es", "fi",
    "fr", "gb", "ge", "gr", "hr", "hu", "ie", "is", "il", "it", "lt", "lu",
    "lv", "me", "mk", "nl", "no", "pl", "pt", "ro", "se", "si", "sk", "ua"
}

SECTOR_MAP = {
    "A": "A_Media_and_Social_Trust",
    "B": "B_Politics_and_Institutions",
    "C": "C_Wellbeing_Health_and_Exclusion",
    "D": "D_Gender_and_Family",
    "E": "E_Module_Justice_or_Climate",
    "F": "F_Sociodemographics_and_Household",
    "G": "G_Work_and_Welfare",
    "H": "H_Schwartz_Human_Values",
    "I": "I_Module_Inequality_or_Covid",
    "J": "J_Interviewer_Paradata",
}

CONTINUOUS_VARS = {
    "agea", "age", "eduyrs", "wkhtot", "wkhct", "yrbrn",
    "dweight", "pspwght", "pweight", "anweight", "prob"
}

LEAKAGE_PREFIXES = ("prtv", "prtcl", "prtmb", "rlgde", "ic")
RLGDN_KEEP = {"rlgdnm", "rlgdnme", "rlgdgr", "rlgatnd", "rlgblg", "rlgblge"}


def download_codebook() -> str:
    if HTML_PATH.exists() and HTML_PATH.stat().st_size > 10_000:
        print(f"[OK] Already exists ({HTML_PATH.stat().st_size:,} bytes)")
        return HTML_PATH.read_text()

    if HTML_PATH.exists():
        HTML_PATH.unlink()

    subprocess.run(
        ["curl", "-L", "-s", "-o", str(HTML_PATH), URL],
        check=True,
    )

    size = HTML_PATH.stat().st_size
    if size < 10_000:
        raise RuntimeError(f"[ERROR] Invalid response ({size} bytes)")

    print(f"[OK] Downloaded ({size:,} bytes)")
    return HTML_PATH.read_text()

In [ ]:
# ==============================================================================
# L0 Step B)  PARSER (Self-contained)
# Define small dicts to help functions separate valuable data from Codebook.html
# ==============================================================================

CORE_SECTOR_FALLBACK = {
    # Section A: Media & Social Trust
    "nwspol": "A_Media_and_Social_Trust", "netusoft": "A_Media_and_Social_Trust",
    "netustm": "A_Media_and_Social_Trust", "ppltrst": "A_Media_and_Social_Trust",
    "pplfair": "A_Media_and_Social_Trust", "pplhlp": "A_Media_and_Social_Trust",
    "rdpol": "A_Media_and_Social_Trust",
    # Section B: Politics & Institutions
    "polintr": "B_Politics_and_Institutions", "psppsgva": "B_Politics_and_Institutions",
    "actrolga": "B_Politics_and_Institutions", "psppipla": "B_Politics_and_Institutions",
    "cptppola": "B_Politics_and_Institutions", "trstprl": "B_Politics_and_Institutions",
    "trstlgl": "B_Politics_and_Institutions", "trstplc": "B_Politics_and_Institutions",
    "trstplt": "B_Politics_and_Institutions", "trstprt": "B_Politics_and_Institutions",
    "trstep": "B_Politics_and_Institutions", "trstun": "B_Politics_and_Institutions",
    "trstsci": "B_Politics_and_Institutions", "vote": "B_Politics_and_Institutions",
    "contplt": "B_Politics_and_Institutions", "wrkprty": "B_Politics_and_Institutions",
    "wrkorg": "B_Politics_and_Institutions", "badge": "B_Politics_and_Institutions",
    "sgnptit": "B_Politics_and_Institutions", "pbldmn": "B_Politics_and_Institutions",
    "pbldmna": "B_Politics_and_Institutions", "bctprd": "B_Politics_and_Institutions",
    "pstplonl": "B_Politics_and_Institutions", "clsprty": "B_Politics_and_Institutions",
    "prtdgcl": "B_Politics_and_Institutions", "mmbprty": "B_Politics_and_Institutions",
    "lrscale": "B_Politics_and_Institutions", "stflife": "B_Politics_and_Institutions",
    "stfeco": "B_Politics_and_Institutions", "stfgov": "B_Politics_and_Institutions",
    "stfdem": "B_Politics_and_Institutions", "stfedu": "B_Politics_and_Institutions",
    "stfhlth": "B_Politics_and_Institutions", "gincdif": "B_Politics_and_Institutions",
    "freehms": "B_Politics_and_Institutions", "hmsfmlsh": "B_Politics_and_Institutions",
    "hmsacld": "B_Politics_and_Institutions", "euftf": "B_Politics_and_Institutions",
    "imsmetn": "B_Politics_and_Institutions", "imdfetn": "B_Politics_and_Institutions",
    "impcntr": "B_Politics_and_Institutions", "imbgeco": "B_Politics_and_Institutions",
    "imueclt": "B_Politics_and_Institutions", "imwbcnt": "B_Politics_and_Institutions",
    "implvdm": "B_Politics_and_Institutions", "volunfp": "B_Politics_and_Institutions",
    "lrnobed": "B_Politics_and_Institutions", "loylead": "B_Politics_and_Institutions",
    "donprty": "B_Politics_and_Institutions",
}


def _extract_table_pairs(data_table_div) -> list[tuple[str, str]]:
    """Extracts raw (value, label) pairs directly from HTML table rows."""
    if not data_table_div:
        return []

    pairs = []
    rows = data_table_div.find_all("tr")
    if rows:
        for tr in rows:
            cells = [td.get_text(" ", strip=True) for td in tr.find_all(["td", "th"])]
            if len(cells) >= 2 and cells[0].lower() != "value":
                pairs.append((cells[0], cells[1]))
    else:
        text = re.sub(r"^Value\s*Category", "", data_table_div.get_text(" ", strip=True))
        pairs = re.findall(
            r"\b(\d{1,6}|[A-Z]{2,3})\b\s+([^\d]+?)(?=\s+\b(?:\d{1,6}|[A-Z]{2,3})\b|$)",
            text,
        )
    return pairs


def _extract_values(data_table_div, var_name: str, var_desc: str) -> tuple[str, dict[str, str]]:
    """
    Returns (range_with_type, labels_dict):
      - range_with_type: '<min> - <max> discrete' | '<min> - <max> cont' | '<N> unique string'
      - labels_dict: {unique_valid_key: category_label} excluding ESS reserved missing codes
    """
    pairs = _extract_table_pairs(data_table_div)
    desc_lower = var_desc.lower()
    reserved_pattern = re.compile(r"^(6+|7+|8+|9+|555555)$")

    # 1. Known continuous survey variables / weights
    if var_name in CONTINUOUS_VARS:
        valid_nums = [
            int(v) for v, _ in pairs
            if v.strip().isdigit() and not (len(v.strip()) >= 2 and reserved_pattern.match(v.strip()))
        ]
        if valid_nums:
            return f"{min(valid_nums)} - {max(valid_nums)} cont", {}
        return "0 - inf cont", {}

    # 2. Pure string / identifier variables without value tables
    if not pairs:
        return "0 unique string", {}

    valid_pairs = []
    for raw_val, raw_lbl in pairs:
        val_str = raw_val.strip()
        lbl_str = raw_lbl.strip()
        if val_str.isdigit():
            if len(val_str) >= 2 and reserved_pattern.match(val_str):
                continue
            valid_pairs.append((int(val_str), lbl_str))
        else:
            valid_pairs.append((val_str, lbl_str))

    if not valid_pairs:
        return "0 unique string", {}

    # 3. Alphanumeric string categorical variables (e.g., cntry ISO codes)
    if any(isinstance(k, str) for k, _ in valid_pairs):
        labels_dict = {str(k): lbl for k, lbl in valid_pairs}
        return f"{len(labels_dict)} unique string", labels_dict

    # 4. Numeric variables: protect 6, 7, 8, 9 when part of 0-10 or 1-10 scales
    max_observed = max(k for k, _ in valid_pairs)
    if max_observed > 10:
        valid_pairs = [(k, lbl) for k, lbl in valid_pairs if k not in {6, 7, 8, 9}]

    if not valid_pairs:
        return "0 - 0 discrete", {}

    nums = [k for k, _ in valid_pairs]
    labels_dict = {str(k): lbl for k, lbl in valid_pairs if lbl}

    is_continuous = (
        len(nums) > 25
        or any(kw in desc_lower for kw in ("year", "hours", "gross pay", "net pay", "number of"))
    )
    var_kind = "cont" if is_continuous else "discrete"

    return f"{min(nums)} - {max(nums)} {var_kind}", labels_dict


def _detect_type(var_name: str, var_desc: str) -> str:
    """Returns: 'country_specific' | 'cross_module' | 'standard'"""
    desc_lower = var_desc.lower()

    if "country-specific" in desc_lower or "country specific" in desc_lower:
        return "country_specific"
    if var_name.startswith(LEAKAGE_PREFIXES):
        return "country_specific"
    if var_name.startswith("rlgdn") and var_name not in RLGDN_KEEP:
        return "country_specific"
    if re.search(r"^[a-z]{4,}[a-z]{2}$", var_name) and var_name[-2:] in COUNTRY_CODES:
        return "country_specific"

    if "cross-module" in desc_lower or "replicated" in desc_lower:
        return "cross_module"

    return "standard"


def parse_codebook(html: str, blacklist: set[str] | None = None) -> dict:
    """
    Parses ESS11 codebook HTML using exact <h3 id="<varname>"> anchors
    and groups variables by questionnaire sector (A-J) or 0_Meta_and_Weights.
    """
    blacklist = blacklist if blacklist is not None else set()
    soup = BeautifulSoup(html, "html.parser")
    ess11 = next(h3 for h3 in soup.find_all("h3") if "ESS11" in h3.get_text())

    toc = ess11.find_next("ul", class_="toc-list")
    toc_entries = {}
    for li in toc.find_all("li"):
        text = li.get_text(strip=True)
        var_name, _, var_desc = text.partition("-")
        toc_entries[var_name.strip()] = var_desc.strip()

    var_blocks = {}
    loc_regex = re.compile(r"Location in questionnaire:\s*([A-Za-z0-9_/-]+)", re.IGNORECASE)

    for h3 in toc.find_all_next("h3"):
        var_id = (h3.get("id") or h3.get_text(strip=True)).strip()
        if var_id not in toc_entries:
            continue

        container = h3.parent if h3.parent and h3.parent.name == "div" else h3
        txt_clean = container.get_text(" ", strip=True)

        sector = CORE_SECTOR_FALLBACK.get(var_id, "0_Meta_and_Weights")
        m_loc = loc_regex.search(txt_clean)
        if m_loc:
            loc_code = m_loc.group(1).strip()
            sec_letter = loc_code[0].upper()
            sector = SECTOR_MAP.get(
                sec_letter,
                f"Other_{sec_letter}" if sec_letter.isalpha() else "0_Meta_and_Weights",
            )

        table_div = container.find("div", class_="data-table") if container.name == "div" else None
        var_blocks[var_id] = {"sector": sector, "table": table_div}

    valid = defaultdict(dict)
    for var_name, var_desc in toc_entries.items():
        if var_name in blacklist:
            continue

        var_type = _detect_type(var_name, var_desc)
        if var_type in ("country_specific", "cross_module"):
            continue

        block = var_blocks.get(var_name, {"sector": "0_Meta_and_Weights", "table": None})
        range_str, labels_dict = _extract_values(block["table"], var_name, var_desc)

        valid[block["sector"]][var_name] = {
            "description": var_desc,
            "range": range_str,
            "labels": labels_dict,
        }

    return dict(valid)

In [ ]:
# ==============================================================================
# L0 Step C) PERSISTING CHECKLIST + BLACKLIST AMPLIFICATION
# Our dynamic selector will need persisted lists in order to select variables properly
# The persisted dictionaries(blacklist and valid_cols) will be the main filter
# for our easy filter  on duckdb query.
# ==============================================================================
def load_blacklist() -> set:
    if BLACKLIST_PATH.exists():
        return set(json.loads(BLACKLIST_PATH.read_text()))
    return set()


def save_blacklist(bl: set):
    BLACKLIST_PATH.write_text(
        json.dumps(sorted(bl), indent=2, ensure_ascii=False)
    )
    print(f"[OK] Saved blacklist ({len(bl)} vars) → {BLACKLIST_PATH}")


def review(valid_cols: dict, step: int = 25) -> dict:
    """
    Iterates sector-by-sector over valid_cols[sector][var_name] wrapping the full
    description across the cell width with a blank line between entries.
    """
    blacklist = load_blacklist()
    total = sum(len(vars_dict) for vars_dict in valid_cols.values())
    term_width = max(100, shutil.get_terminal_size((120, 24)).columns)
    prefix_len = 49  # "  " (2) + name (14) + " | " (3) + range (20) + " | labels:XXX | " (10) -> ~52 chars
    desc_width = max(50, term_width - 54)

    print(f"\n{'='*term_width}")
    print(f"  REVIEW: {total} variables across {len(valid_cols)} sectors, step={step}")
    print(f"  Current blacklist: {len(blacklist)} vars")
    print(f"  Copy line, DELETE unwanted variables, paste. (s = keep all without changes)")
    print(f"{'='*term_width}\n")

    for sector in sorted(valid_cols.keys()):
        vars_dict = valid_cols[sector]
        sector_names = [n for n in vars_dict.keys() if n not in blacklist]
        if not sector_names:
            continue

        print(f"\n>>> SECTOR: {sector} ({len(sector_names)} vars)")
        for i in range(0, len(sector_names), step):
            batch = sector_names[i:i + step]
            print(f"\n--- [{i+1}-{i+len(batch)}/{len(sector_names)}] ({sector}) ---\n")
            for name in batch:
                meta = vars_dict[name]
                n_lbl = len(meta["labels"])
                header = f"  {name:14s} | {meta['range']:20s} | labels:{n_lbl:3d} | "
                wrapped_desc = textwrap.fill(
                    meta["description"],
                    width=desc_width,
                    subsequent_indent=" " * len(header),
                )
                print(f"{header}{wrapped_desc}\n")

            print(f"  COPY: {','.join(batch)}")
            user_input = input("\n>>> Paste here (or 's'): ").strip()

            if not user_input or user_input == "s":
                continue

            kept = {x.strip() for x in user_input.split(",") if x.strip()}
            to_exclude = [n for n in batch if n not in kept and n not in blacklist]

            if to_exclude:
                blacklist.update(to_exclude)
                print(f"  [X] {len(to_exclude)} -> blacklist: {', '.join(to_exclude)}")
                save_blacklist(blacklist)
            else:
                print("  [OK] No changes")

    filtered_valid = {
        sector: {k: v for k, v in vars_dict.items() if k not in blacklist}
        for sector, vars_dict in valid_cols.items()
    }
    filtered_valid, blacklist = coalesce_valid_cols(filtered_valid, blacklist)
    save_blacklist(blacklist)

    remaining_count = sum(len(v) for v in filtered_valid.values())
    print(f"\n{'='*term_width}")
    print(f"  FINAL: {remaining_count} valid / {len(blacklist)} discarded / {total} parsed")
    print(f"{'='*term_width}")
    return filtered_valid
    
def load_or_build_metadata(
    force_rebuild: bool = False,
    run_review: bool = False,
    step: int = 25,
) -> tuple[dict, set[str]]:
    """
    If valid_cols.json and blacklist.json already exist in TABLES and force_rebuild=False,
    skip HTML parsing and interactive revision (cache hit).
    """
    blacklist = load_blacklist()

    if DICT_PATH.exists() and DICT_PATH.stat().st_size > 100 and not force_rebuild:
        valid_cols = json.loads(DICT_PATH.read_text())
        total_vars = (
            sum(len(v) for v in valid_cols.values())
            if all(isinstance(v, dict) and "description" not in v for v in valid_cols.values())
            else len(valid_cols)
        )
        print(f"[SKIP] Existing metadata found -> {DICT_PATH} ({total_vars} valid vars, {len(blacklist)} blacklisted)")

        if run_review:
            valid_cols = review(valid_cols, step=step)
            DICT_PATH.write_text(json.dumps(valid_cols, indent=2, ensure_ascii=False))

        return valid_cols, blacklist

    # Full execution only if JSONs do not exist or if force_rebuild=True
    html = download_codebook()
    valid_cols = parse_codebook(html, blacklist=blacklist)
    DICT_PATH.write_text(json.dumps(valid_cols, indent=2, ensure_ascii=False))
    print(f"[OK] Built valid_cols -> {DICT_PATH}")

    if run_review:
        valid_cols = review(valid_cols, step=step)
        DICT_PATH.write_text(json.dumps(valid_cols, indent=2, ensure_ascii=False))

    return valid_cols, blacklist
COALESCE_GROUPS = {
    "cntbrth":  ["cntbrtha", "cntbrthb", "cntbrthc"],
    "ctzship":  ["ctzshipa", "ctzshipb", "ctzshipc"],
    "fbrncnt":  ["fbrncnt", "fbrncnta", "fbrncntb"],
    "mbrncnt":  ["mbrncnt", "mbrncnta", "mbrncntb"],
    "lnghom1":  ["lnghom1", "lnghoma"],
    "lnghom2":  ["lnghom2", "lnghomb"],
    "anctry1":  ["anctry1", "anctrya1"],
    "anctry2":  ["anctry2", "anctrya2"],
    "dvrcdev":  ["dvrcdev", "dvrcdeva"],
    "edulvl":   ["edulvlb", "edulvla"],
    "edulvlf":  ["edulvlfb", "edulvlfa"],
    "edulvlm":  ["edulvlmb", "edulvlma"],
    "edulvlp":  ["edulvlpb", "edulvlpa"],
    "hincsrc":  ["hincsrca", "hincsrc"],
    "hinctnt":  ["hinctnta", "hinctnt"],
    "lvghw":    ["lvghwa", "lvghw"],
    "lvgptn":   ["lvgptna", "lvgptn"],
    "lvgptne":  ["lvgptnea", "lvgptne"],
    "marital":  ["maritalb", "maritala"],
    "occf14":   ["occf14b", "occf14a", "occf14"],
    "occm14":   ["occm14b", "occm14a", "occm14"],
    "wrkctr":   ["wrkctra", "wrkctr"],
    "wkdcorg":  ["wkdcorga", "wkdcorg"],
    "dcsfwrk":  ["dcsfwrka", "dcsfwrk"],
    "dfcnswk":  ["dfcnswka", "dfcnswk"],
    "dsgrmny":  ["dsgrmnya", "dsgrmny"],
    "grspay":   ["grspaya", "grspay"],
    "payprd":   ["payprda", "payprd"],
    "pfmfdjb":  ["pfmfdjba", "pfmfdjb"],
    "yrspdwk":  ["yrspdwka", "yrspdwk"],
    "impdiff":  ["impdiffa", "impdiff"],
    "impenv":   ["impenva", "impenv"],
    "impfun":   ["impfuna", "impfun"],
    "impsafe":  ["impsafea", "impsafe"],
    "imptrad":  ["imptrada", "imptrad"],
    "ipadvnt":  ["ipadvnta", "ipadvnt"],
    "ipbhprp":  ["ipbhprpa", "ipbhprp"],
    "ipcrtiv":  ["ipcrtiva", "ipcrtiv"],
    "ipfrule":  ["ipfrulea", "ipfrule"],
    "ipgdtim":  ["ipgdtima", "ipgdtim"],
    "ipmodst":  ["ipmodsta", "ipmodst"],
    "iprspot":  ["iprspota", "iprspot"],
    "ipshabt":  ["ipshabta", "ipshabt"],
    "ipstrgv":  ["ipstrgva", "ipstrgv"],
    "ipudrst":  ["ipudrsta", "ipudrst"],
    "clmthgt":  ["clmthgt2", "clmthgt1"],
    "panpriph": ["panpriph", "govpriph"],
    "panmonpb": ["panmonpb", "govmonpb"],
    **{f"rship{i}": [f"rshipa{i}", f"rship{i}"] for i in range(2, 16)},
}


def coalesce_valid_cols(valid_cols: dict, blacklist: set[str]) -> tuple[dict, set[str]]:
    """
    If the user retained at least one variant of a coalescible pair in valid_cols,
    it binds them under the canonical key with 'coalesce_from' and removes duplicates.
    """
    # Locate in which sector each conserved variable is
    var_location = {}
    for sector, vars_dict in valid_cols.items():
        for vname, meta in vars_dict.items():
            var_location[vname] = (sector, meta)

    for canonical, members in COALESCE_GROUPS.items():
        kept_members = [m for m in members if m in var_location]
        if not kept_members:
            continue  # The user discarded all variants of the group

        # Take metadata and sector of the conserved variant
        sector, rep_meta = var_location[kept_members[0]]

        # Delete individual variants from the dictionary so as not to duplicate columns
        for m in kept_members:
            m_sec, _ = var_location[m]
            valid_cols[m_sec].pop(m, None)

        # Remove the sister variants from blacklist just in case 
        # the user deleted one thinking it was duplicating
        blacklist.difference_update(members)

        # Insert unified canonical entry
        valid_cols[sector][canonical] = {
            "description": rep_meta["description"],
            "range": rep_meta["range"],
            "labels": rep_meta["labels"],
            "coalesce_from": members,
        }

    # Clean sectors that have been left empty after uniting
    cleaned_valid = {sec: vdict for sec, vdict in valid_cols.items() if vdict}
    return cleaned_valid, blacklist

In [ ]:
# ==============================================================================
# L0 Step D) Calling functions and selecting mode of use on MAIN
# ==============================================================================
if __name__ == "__main__":
    valid_cols, blacklist = load_or_build_metadata(
        force_rebuild=False,  # True -> ignores cache and regenerates from ESSCodebook.html
        run_review=False,     # True -> open interactive review on the rest
        step=25,
    )

    total_valid = (
        sum(len(v) for v in valid_cols.values())
        if all(isinstance(v, dict) and "description" not in v for v in valid_cols.values())
        else len(valid_cols)
    )
    print(f"[READY] Layer 0-1 metadata loaded -> {total_valid} valid vars | {len(blacklist)} blacklisted")

In [ ]:
# ==============================================================================
# STEP 0-1.C) PERSIST L0 DICTIONARIES INTO DUCKDB (./temp/ess11.duckdb)
# ==============================================================================
import json
import re
from datetime import datetime, timezone
import duckdb
import pandas as pd

DUCKDB_PATH = TABLES / "ess11.duckdb"


def _parse_range_parts(range_str: str) -> tuple[str, float | None, float | None]:
    """Extracts (var_kind, min_val, max_val) from range string like '0 - 10 discrete'."""
    if "string" in range_str:
        return "string", None, None
    kind = "cont" if "cont" in range_str else "discrete"
    m = re.search(r"(-?\d+(?:\.\d+)?)\s*-\s*(-?\d+(?:\.\d+)?)", range_str)
    if m:
        return kind, float(m.group(1)), float(m.group(2))
    return kind, None, None


def persist_l0_to_duckdb(
    valid_cols: dict,
    blacklist: set[str],
    db_path: Path = DUCKDB_PATH,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Persists valid_cols and blacklist dictionaries directly into DuckDB
    inside ./temp/ess11.duckdb so S0.1_DATAEXTRACT_L1 queries them natively.
    """
    valid_rows = []
    is_nested = all(isinstance(v, dict) and "description" not in v for v in valid_cols.values())

    if is_nested:
        for sector, vars_dict in valid_cols.items():
            for var_name, meta in vars_dict.items():
                kind, min_v, max_v = _parse_range_parts(meta.get("range", ""))
                valid_rows.append({
                    "var_name": var_name,
                    "sector": sector,
                    "description": meta.get("description", ""),
                    "range_str": meta.get("range", ""),
                    "var_kind": kind,
                    "min_val": min_v,
                    "max_val": max_v,
                    "coalesce_from_json": json.dumps(meta.get("coalesce_from", []), ensure_ascii=False),
                    "labels_json": json.dumps(meta.get("labels", {}), ensure_ascii=False),
                })
    else:
        for var_name, meta in valid_cols.items():
            kind, min_v, max_v = _parse_range_parts(meta.get("range", ""))
            valid_rows.append({
                "var_name": var_name,
                "sector": meta.get("sector", "0_Meta_and_Weights"),
                "description": meta.get("description", ""),
                "range_str": meta.get("range", ""),
                "var_kind": kind,
                "min_val": min_v,
                "max_val": max_v,
                "coalesce_from_json": json.dumps(meta.get("coalesce_from", []), ensure_ascii=False),
                "labels_json": json.dumps(meta.get("labels", {}), ensure_ascii=False),
            })

    df_valid = pd.DataFrame(valid_rows).sort_values(["sector", "var_name"]).reset_index(drop=True)
    df_black = pd.DataFrame({
        "var_name": sorted(blacklist),
        "excluded_in_layer": "L0",
    })

    df_manifest = pd.DataFrame([{
        "layer": "L0",
        "ess_round": "ESS11",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "n_valid_vars": len(df_valid),
        "n_blacklisted_vars": len(df_black),
        "n_coalesce_rules": int((df_valid["coalesce_from_json"] != "[]").sum()),
    }])

    with duckdb.connect(str(db_path)) as con:
        con.execute("CREATE OR REPLACE TABLE l0_valid_cols AS SELECT * FROM df_valid")
        con.execute("CREATE OR REPLACE TABLE l0_blacklist AS SELECT * FROM df_black")
        con.execute("CREATE OR REPLACE TABLE l0_manifest AS SELECT * FROM df_manifest")

        summary = con.execute("""
            SELECT sector, COUNT(*) AS n_vars
            FROM l0_valid_cols
            GROUP BY sector
            ORDER BY sector
        """).df()

    print(f"[OK] L0 persisted in DuckDB -> {db_path}")
    print(f"     - l0_valid_cols : {len(df_valid)} rows ({df_manifest['n_coalesce_rules'].iloc[0]} coalesce rules)")
    print(f"     - l0_blacklist  : {len(df_black)} rows")
    display(summary)
    return df_valid, df_black




In [ ]:
# ==============================================================================
# 4. MAIN (Final L0 Execution & DuckDB Persistence)
# ==============================================================================
if __name__ == "__main__":
    valid_cols, blacklist = load_or_build_metadata(
        force_rebuild=False,
        run_review=False,
        step=25,
    )

    df_valid_meta, df_black_meta = persist_l0_to_duckdb(valid_cols, blacklist, DUCKDB_PATH)
    print(f"[READY] S0.0_DATAEXTRACT_L0 finished -> {DUCKDB_PATH.name} ready for S0.1_DATAEXTRACT_L1")